#Bibliotecas


In [ ]:
!git clone https://github.com/elphick/geometallurgy.git
%cd geometallurgy
!pip install .

"""
O collab não tem em seu acervo o elhpcik geometallurgy sendo necessário clonar o
repositório pelo github
"""
%cd ..
!pip install pandas pandasai openai matplotlib plotly scikit-learn ipython


import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from pandasai import SmartDataframe
from pandasai.llm import OpenAI
from geometallurgy.recovery import recovery_mass, recovery_metal
import warnings
warnings.filterwarnings('ignore')

# Entrada por upload(provisório)
#### Célula deverá sera tualziada ao usar um compilador local.

In [ ]:
from google.colab import files
uploaded = files.upload()

filename = next(iter(uploaded))
df = pd.read_csv(filename)

# Dicionário de colunas feito para talvez futuramente aplicar aprendizado supervisonado
column_dic = {
    "codigo": "Identificador da amostra ou ensaio",
    "data": "Data do ensaio ou coleta",
    "usuario": "Responsável pelo ensaio",
    "amostra": "Nome ou código da amostra analisada",
    "localizacao": "Posição da amostra (ex: furo de sondagem, coordenada, etc)",
    "granulometria(p80)": "P80 da amostra, representando a malha de liberação (µm)",
    "massa_alim": "Massa da alimentação no ensaio (g)",
    "massa_conc": "Massa do concentrado (g)",
    "massa_rej": "Massa do rejeito (g)",
    "rec_massica": "Recuperação massiva (%)",
    "rec_metalurgica": "Recuperação metalúrgica do metal de interesse (%)",
    "cu_alim": "Teor de cobre na alimentação (%)",
    "cu_conc": "Teor de cobre no concentrado (%)",
    "cu_rej": "Teor de cobre no rejeito (%)"
}

# Recalculo 1
#### Recacalculo feito escrito em ordem natural para aferição dos resultados da recuperação metalurgica onde iremos aplicar para um padrão de confiabildiade de cada amostra.

In [ ]:
df['rec_massica_calc'] = (df['massa_conc'] / df['massa_alim']) * 100
df['rec_metalurgica_calc'] = (df['cu_conc'] * df['massa_conc']) / (df['cu_alim'] * df['massa_alim']) * 100

df['delta_rec_massica'] = df['rec_massica'] - df['rec_massica_calc']
df['delta_rec_metalurgica'] = df['rec_metalurgica'] - df['rec_metalurgica_calc']


print("\n Aferição de recalculo")
display(df.head())

discrep_massica = df[df['delta_rec_massica'].abs() > 1]
discrep_metalurgica = df[df['delta_rec_metalurgica'].abs() > 1]

# Recalculo 2
#### Célula que mantém o objetivo original de aferição de validação dos dados de recuperação metalurgica porem agora usando definições existente no elphick geometallurgy para teste

In [ ]:
def recalcular_recuperacao(df):
    df["rec_metalurgica_calc"] = RecoveryCalculator.recovery(
        feed_assay=df["cu_alim"],
        concentrate_assay=df["cu_conc"],
        concentrate_mass=df["massa_conc"],
        feed_mass=df["massa_alim"]
    )
    return df

def validar_dados(df):
    erro_rec = (df["rec_metalurgica"] - df["rec_metalurgica_calc"]).abs()
    print("Erros médios entre recuperação declarada e calculada:", erro_rec.mean())
    return erro_rec

# Aprendizado (TESTE)
#### Célula em que deverá operar os modelos de aprendizagem simples não supervisionados

In [ ]:
def model_predit(df):
    features = ["cu_alim", "massa_alim", "massa_conc", "cu_conc", "granulometria(p80)"]
    target = "rec_metalurgica_calc"

    df = df.dropna(subset=features + [target])
    X = df[features]
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

    model = RandomForestRegressor()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)

    print(f"Erro quadrático médio (MSE): {mse:.4f}")
    return model

def clusterizacao(df, n_clusters=3):
    cluster_data = df[["cu_alim", "cu_conc", "granulometria(p80)"]].dropna()
    kmeans = KMeans(n_clusters=n_clusters)
    df["cluster"] = kmeans.fit_predict(cluster_data)
    return df

def plot_clusters(df):
    fig = px.scatter(df, x="cu_alim", y="cu_conc", color="cluster", title="Clusterização por teor")
    fig.show()